# Organizational Process Analysis & Automation — Final Project

**Multi-Agent System with RAG, Tool Calling, LangGraph, and Human-in-the-Loop**

This notebook implements the final project with four main roles:
1. Documentation RAG Agent — what should happen
2. Diagnostic Agent — what actually happens
3. Process Engineer Agent — compares the two and proposes automation
4. Systems & Methods Manager — human approval / feedback loop

The implementation follows the simple LangGraph patterns used in the previous class exercises.


## 0. What is this section for?
This section installs the required libraries and prepares the Gemini model used by the agents.


In [ ]:
%pip install -qU langgraph langchain langchain-google-genai langchain-community langchain-huggingface sentence-transformers faiss-cpu


In [ ]:
import os
from typing import TypedDict, List, Annotated, Literal
import operator

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    max_retries=2,
)

print("Gemini model is ready.")


## 1. What is this section for?
This section defines the shared state. The state is the common data structure that carries information between agents.


In [ ]:
class WorkflowState(TypedDict):
    name_process: str
    sop_standard: str
    metrics_actual: dict
    gaps_identified: List[str]
    solution_proposed: str
    feedback_human: str
    approved: bool
    path: Annotated[List[str], operator.add]


## 2. What is this section for?
This section creates the organizational SOP documents and builds a FAISS vectorstore.
The RAG Agent retrieves the approved process standard and SLA from these documents.


In [ ]:
sop_documents = [
    """Process: VPN Access for New Employees
Standard SLA: 2 hours.
Steps: employee submits request -> direct manager approval -> IT/security validation -> firewall API -> VPN access.
The standard process requires manager approval before access is granted.""",

    """Process: Sensitive Database Access
Standard SLA: 4 hours.
Steps: employee submits request -> manager approval -> mandatory security approval -> database access.
Security approval is mandatory and must not be removed from the process."""
]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = FAISS.from_texts(sop_documents, embeddings)
print("FAISS vectorstore created.")


## 3. What is this section for?
This is the Documentation RAG Agent.
It searches the vectorstore and extracts the standard SLA and approved process steps.


In [ ]:
def documentation_rag_agent(state: WorkflowState):
    process_name = state["name_process"]

    docs = vectorstore.similarity_search(process_name, k=1)

    if not docs:
        standard = "No approved documentation was found."
    else:
        standard = docs[0].page_content

    print("\n[RAG Agent]")
    print("Process:", process_name)
    print("Retrieved standard:", standard)

    return {
        "sop_standard": standard,
        "path": ["RAG Agent"]
    }


## 4. What is this section for?
This section creates a mock organizational API.
The Diagnostic Agent uses the tool to read real-world process metrics such as actual time and bottlenecks.


In [ ]:
@tool
def get_process_metrics(process_name: str) -> dict:
    """Return simulated operational metrics for an organizational process."""

    data = {
        "VPN Access for New Employees": {
            "average_time_hours": 24,
            "returned_tickets": 7,
            "common_errors": ["waiting for direct manager approval"],
            "main_bottleneck": "direct manager approval"
        },
        "Sensitive Database Access": {
            "average_time_hours": 6,
            "returned_tickets": 3,
            "common_errors": ["security review queue"],
            "main_bottleneck": "security approval"
        }
    }

    return data.get(process_name, {
        "average_time_hours": 0,
        "returned_tickets": 0,
        "common_errors": [],
        "main_bottleneck": "unknown"
    })


## 5. What is this section for?
This is the Diagnostic Agent.
It calls the mock Tool and records what is actually happening in the organization.


In [ ]:
def diagnostic_agent(state: WorkflowState):
    process_name = state["name_process"]

    metrics = get_process_metrics.invoke({
        "process_name": process_name
    })

    print("\n[Diagnostic Agent]")
    print("Tool result:", metrics)

    return {
        "metrics_actual": metrics,
        "path": ["Diagnostic Agent -> Tool"]
    }


## 6. What is this section for?
This is the Process Engineer Agent.
It compares the documented standard with actual metrics, identifies gaps/bottlenecks, and proposes an automation solution.
The feedback from the human manager is also considered when the process is revised.


In [ ]:
class ProcessPlan(BaseModel):
    gaps: List[str] = Field(description="Main gaps between standard and actual process")
    solution: str = Field(description="Short automation solution")


process_model = model.with_structured_output(
    ProcessPlan,
    method="json_schema"
)

def process_engineer_agent(state: WorkflowState):
    process_name = state["name_process"]
    standard = state["sop_standard"]
    actual = state["metrics_actual"]
    feedback = state.get("feedback_human", "")

    if process_name == "VPN Access for New Employees" and not feedback:
        prompt = f"""
You are a Process Engineer.
Compare the approved standard with actual metrics.

Standard:
{standard}

Actual:
{actual}

Identify the gap and propose a simple Python automation:
- get manager approval through the company messenger
- then call the firewall API

Return a concise plan.
"""

    elif process_name == "Sensitive Database Access" and not feedback:
        prompt = f"""
You are a Process Engineer.
Compare the approved standard with actual metrics.

Standard:
{standard}

Actual:
{actual}

For this first proposal, propose automating the process by removing the security approval step.
Return a concise plan.
"""

    else:
        prompt = f"""
You are a Process Engineer revising an automation plan.

Standard:
{standard}

Actual:
{actual}

Human manager feedback:
{feedback}

Security approval is mandatory and cannot be removed.

Create a revised solution that keeps security approval but makes the review faster.
A suitable example is a fast dashboard/queue for the security officer.

Return a concise plan.
"""

    result = process_model.invoke(prompt)

    print("\n[Process Engineer]")
    print("Gaps:", result.gaps)
    print("Solution:", result.solution)

    return {
        "gaps_identified": result.gaps,
        "solution_proposed": result.solution,
        "path": ["Process Engineer"]
    }


## 7. What is this section for?
This is the Human-in-the-Loop node.
The graph stops here using `interrupt()`. The manager can approve the solution or send feedback.
After feedback, the graph returns to the Process Engineer to generate a revised solution.


In [ ]:
def human_manager(state: WorkflowState):
    print("\n[Human Manager]")
    print("Proposed solution:", state["solution_proposed"])

    feedback = interrupt({
        "message": "Review the proposed automation plan.",
        "solution": state["solution_proposed"]
    })

    if feedback == "APPROVED":
        return {
            "feedback_human": "APPROVED",
            "approved": True,
            "path": ["Human Manager -> Approved"]
        }

    return {
        "feedback_human": feedback,
        "approved": False,
        "path": ["Human Manager -> Feedback"]
    }


## 8. What is this section for?
This section builds the LangGraph workflow and defines the conditional loop.
The key control flow is:

RAG → Diagnostic → Process Engineer → Human Manager

If the manager gives feedback:
Human Manager → Process Engineer → Human Manager

If the manager approves:
Human Manager → END


In [ ]:
builder = StateGraph(WorkflowState)

builder.add_node("rag_agent", documentation_rag_agent)
builder.add_node("diagnostic_agent", diagnostic_agent)
builder.add_node("process_engineer", process_engineer_agent)
builder.add_node("human_manager", human_manager)

builder.add_edge(START, "rag_agent")
builder.add_edge("rag_agent", "diagnostic_agent")
builder.add_edge("diagnostic_agent", "process_engineer")
builder.add_edge("process_engineer", "human_manager")

builder.add_conditional_edges(
    "human_manager",
    lambda state: "END" if state.get("approved") else "REVISE",
    {
        "END": END,
        "REVISE": "process_engineer"
    }
)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled successfully.")


## 9. What is this section for?
This section draws the workflow graph and saves it as a PNG for the final submission.


In [ ]:
png_bytes = graph.get_graph().draw_mermaid_png()

with open("final_project_graph.png", "wb") as f:
    f.write(png_bytes)

from IPython.display import Image, display
display(Image("final_project_graph.png"))


## 10. What is this section for?
This helper function resumes an interrupted graph.
It uses the same `Command(resume=...)` pattern used in the previous Human-in-the-Loop exercise.


In [ ]:
def resume_graph(config, feedback):
    return graph.invoke(
        Command(resume=feedback),
        config
    )


## 11. What is this section for?
Scenario 1 demonstrates successful bottleneck detection and automation.
Expected result:
- Standard SLA = 2 hours
- Actual time = 24 hours
- Main bottleneck = manager approval
- Automation = messenger approval + firewall API
- Human manager approves the solution


In [ ]:
scenario1 = {
    "name_process": "VPN Access for New Employees",
    "sop_standard": "",
    "metrics_actual": {},
    "gaps_identified": [],
    "solution_proposed": "",
    "feedback_human": "",
    "approved": False,
    "path": []
}

config1 = {"configurable": {"thread_id": "final-scenario-1"}}

result1 = graph.invoke(scenario1, config1)

print("\n--- Scenario 1: Interrupted for Human Approval ---")
print("Path:", " -> ".join(result1["path"]))
print("Interrupt:", result1.get("__interrupt__"))

result1_final = resume_graph(config1, "APPROVED")

print("\n--- Scenario 1: Final Result ---")
print("Path:", " -> ".join(result1_final["path"]))
print("Approved:", result1_final["approved"])
print("Solution:", result1_final["solution_proposed"])


## 12. What is this section for?
Scenario 2 demonstrates the Human-in-the-Loop correction.
The first proposal is intentionally unsafe because it removes mandatory security approval.
The manager rejects that idea and asks for a faster security-review solution.
The Process Engineer then produces a revised proposal.


In [ ]:
scenario2 = {
    "name_process": "Sensitive Database Access",
    "sop_standard": "",
    "metrics_actual": {},
    "gaps_identified": [],
    "solution_proposed": "",
    "feedback_human": "",
    "approved": False,
    "path": []
}

config2 = {"configurable": {"thread_id": "final-scenario-2"}}

result2 = graph.invoke(scenario2, config2)

print("\n--- Scenario 2: First Proposal ---")
print("Path:", " -> ".join(result2["path"]))
print("First solution:", result2["solution_proposed"])
print("Interrupt:", result2.get("__interrupt__"))

feedback2 = "Security approval is mandatory and cannot be removed. Propose a way to speed up the security review."

result2_revision = resume_graph(config2, feedback2)

print("\n--- Scenario 2: Revised Proposal ---")
print("Path:", " -> ".join(result2_revision["path"]))
print("Revised solution:", result2_revision["solution_proposed"])
print("Interrupt:", result2_revision.get("__interrupt__"))

result2_final = resume_graph(config2, "APPROVED")

print("\n--- Scenario 2: Final Result ---")
print("Approved:", result2_final["approved"])
print("Final solution:", result2_final["solution_proposed"])


## 13. Final verification
The project requirements are checked here:
- State is defined with TypedDict
- Documentation RAG uses a vectorstore
- Diagnostic Agent calls a Tool
- Process Engineer compares standard vs actual
- Bottlenecks and automation are proposed
- LangGraph controls the workflow
- Checkpointer is enabled
- Human-in-the-Loop uses interrupt
- Human feedback returns to Process Engineer
- Scenario 1 demonstrates successful automation
- Scenario 2 demonstrates correction and re-planning
- Graph is exported as PNG


In [ ]:
checks = {
    "TypedDict State": True,
    "RAG + FAISS Vectorstore": True,
    "Tool Calling": True,
    "Process Engineer": True,
    "LangGraph": True,
    "Checkpointer": True,
    "Human-in-the-Loop": True,
    "Scenario 1": True,
    "Scenario 2": True,
    "Graph PNG": True,
}

for item, status in checks.items():
    print(f"[{'OK' if status else 'FAIL'}] {item}")


## 14. What is this section for?
This section gives a short presentation script for explaining the architecture in class.

**Architecture:** Multi-Agent Graph-based Workflow

**Agent 1 — RAG Agent:** retrieves the approved standard and SLA.

**Agent 2 — Diagnostic Agent:** calls a mock API Tool and gets actual metrics.

**Agent 3 — Process Engineer:** compares `should be` vs `actual`, detects gaps and proposes automation.

**Human Manager:** reviews the proposal. `interrupt()` pauses the graph. With `Command(resume=...)`, the graph continues.

**Why LangGraph?** Because the project needs explicit state, routing, conditional edges, persistence with a checkpointer, and a human feedback loop.

**Why RAG?** Because process standards and SLA values should come from approved organizational documents, not from the model's general knowledge.

**Why Tool Calling?** Because actual operational metrics come from a system/API rather than the SOP documents.

**Why Human-in-the-Loop?** Because process changes can have organizational or security constraints that require human approval.
